In [1]:
import sys
import os
import openseespy.opensees as ops
import numpy as np
import pandas as pd
from math import pi, cos, cosh, ceil, log
from scipy.stats import norm, gumbel_r

# Cross-platform path handling
current_dir = os.getcwd()
sys.path.append(os.path.join(current_dir, 'SRC', 'interpreter'))
sys.path.append(os.path.join(current_dir, 'PyPonding'))

from PyPonding.structures.wide_flange import wf
from libdenavit.section.database import wide_flange_database

In [2]:
# Units
inch = 1.0
kip = 1.0
hr = 1.0
lb = kip/1000.0
ft = 12.0*inch
ksi = kip/inch**2
psf = lb/ft**2
pcf = psf/ft
g = 386.4*inch/(hr/3600)**2
gamma = 62.4*pcf

In [3]:
# Load rainfall data
df_mean = pd.read_csv('mean_intensity_conf_interval.csv', index_col='Dur (hr)')
rain_rate = df_mean.loc['x0.25hr', '100yr'] * (inch/hr)

# Section properties (W8X18)
shape_name = 'W8X18'
L = 30.0 * ft
tw = 10.0 * ft
Ss = 40.0 * ft
cd, ws = 0.6, 12.0 * inch

wfs = wide_flange_database[shape_name]
wf_section = wf(wfs['d'], wfs['tw'], wfs['bf'], wfs['tf'], 50.0*ksi, 29000.0*ksi, 29.0*ksi)
wf_section.material_type = 'Hardening'
wf_section.wD = (10.0 * lb / ft**2) * tw
wf_section.L, wf_section.TW, wf_section.gamma = L, tw, gamma

In [ ]:
# Calculate Design Depth (z)
As = Ss * L
q = rain_rate * As
design_z = 2.0 + (1.5 * q / (cd * ws * (2 * g)**0.5))**(2.0/3.0)

# Case 1: No Slope
wf_section.zj = 0.0
sr_flat = wf_section.SR_ratio_for_a_given_z(design_z)
print(f"Flat Case (z={design_z:.3f}):\n{sr_flat}")

# Case 2: With Slope (0.25/12)
wf_section.zj = (0.25/12.0) * L
sr_sloped = wf_section.SR_ratio_for_a_given_z(design_z)
print(f"Sloped Case (z={design_z:.3f}):\n{sr_sloped}")


 Fy = 50.000 ksi Zz = 16.748 in^3 capacity = 753.659 kip-in
Level =  2.9904, Strength Ratio =  1.5310865 (Moment at x/L = 0.500)
Flat Case (z=2.990):
1.5310864952979544

 Fy = 50.000 ksi Zz = 16.748 in^3 capacity = 753.659 kip-in
Level =  2.9904, Strength Ratio =  0.4489235 (Moment at x/L = 0.450)
Sloped Case (z=2.990):
0.4489235234960561


In [5]:
# Analysis and Results
As = Ss * L
q = rain_rate * As
# Calculate design water depth z
design_z = 2.0 + (1.5 * q / (cd * ws * (2 * g)**0.5))**(2.0/3.0)

# Scenario A: Flat
wf_section.zj = 0.0
sr_flat = wf_section.SR_ratio_for_a_given_z(design_z)
print(f"Flat Case SR (z={design_z:.3f}):\n{sr_flat}")

# Scenario B: Sloped (0.25/12)
wf_section.zj = (0.25/12.0) * L
sr_sloped = wf_section.SR_ratio_for_a_given_z(design_z)
print(f"Sloped Case SR (z={design_z:.3f}):\n{sr_sloped}")


 Fy = 50.000 ksi Zz = 16.748 in^3 capacity = 753.659 kip-in
Level =  2.9904, Strength Ratio =  1.5310865 (Moment at x/L = 0.500)
Flat Case SR (z=2.990):
1.5310864952979544

 Fy = 50.000 ksi Zz = 16.748 in^3 capacity = 753.659 kip-in
Level =  2.9904, Strength Ratio =  0.4489235 (Moment at x/L = 0.450)
Sloped Case SR (z=2.990):
0.4489235234960561
